# บทที่ 8: Convolutional Neural Networks (CNN)

ใน Notebook นี้ เราจะ implement Convolution Operation, Pooling และศึกษาโครงสร้างของ CNN

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from scipy import signal

np.random.seed(42)

## 2. Convolution Operation

In [ ]:
def conv2d(image, kernel, stride=1, padding=0):
    """
    2D Convolution operation
    
    Parameters:
    - image: Input image (H x W)
    - kernel: Filter/Kernel (kH x kW)
    - stride: Step size
    - padding: Zero-padding
    """
    # Add padding
    if padding > 0:
        image = np.pad(image, padding, mode='constant')
    
    h, w = image.shape
    kh, kw = kernel.shape
    
    # Output dimensions
    out_h = (h - kh) // stride + 1
    out_w = (w - kw) // stride + 1
    
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            # Extract region
            region = image[i*stride:i*stride+kh, j*stride:j*stride+kw]
            # Convolution (element-wise multiply and sum)
            output[i, j] = np.sum(region * kernel)
            
    return output

# Test with simple image
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

# Edge detection kernel
kernel = np.array([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
])

print("=== Original Image ===")
print(image)

print("\n=== Kernel (Vertical Edge Detection) ===")
print(kernel)

output = conv2d(image, kernel)
print("\n=== Convolution Output ===")
print(output)

## 3. Output Size Calculation

In [ ]:
def calculate_output_size(input_size, kernel_size, stride=1, padding=0):
    """
    Calculate output size after convolution
    
    Formula: output_size = (input_size + 2*padding - kernel_size) / stride + 1
    """
    output_size = (input_size + 2*padding - kernel_size) // stride + 1
    return output_size

# Examples
print("=== Output Size Calculations ===")
print(f"Input: 32×32, Kernel: 3×3, Stride: 1, Padding: 0")
print(f"Output: {calculate_output_size(32, 3, 1, 0)}×{calculate_output_size(32, 3, 1, 0)}")

print(f"\nInput: 32×32, Kernel: 3×3, Stride: 1, Padding: 1")
print(f"Output: {calculate_output_size(32, 3, 1, 1)}×{calculate_output_size(32, 3, 1, 1)}")

print(f"\nInput: 32×32, Kernel: 2×2, Stride: 2, Padding: 0")
print(f"Output: {calculate_output_size(32, 2, 2, 0)}×{calculate_output_size(32, 2, 2, 0)}")

## 4. Pooling Operations

In [ ]:
def max_pool2d(image, pool_size=2, stride=2):
    """Max Pooling"""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.max(region)
            
    return output

def avg_pool2d(image, pool_size=2, stride=2):
    """Average Pooling"""
    h, w = image.shape
    out_h = (h - pool_size) // stride + 1
    out_w = (w - pool_size) // stride + 1
    
    output = np.zeros((out_h, out_w))
    
    for i in range(out_h):
        for j in range(out_w):
            region = image[i*stride:i*stride+pool_size, j*stride:j*stride+pool_size]
            output[i, j] = np.mean(region)
            
    return output

# Test
image = np.array([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
])

print("=== Original Image ===")
print(image)

print("\n=== Max Pooling (2×2) ===")
print(max_pool2d(image))

print("\n=== Average Pooling (2×2) ===")
print(avg_pool2d(image))

## 5. Edge Detection Example

In [ ]:
# Create a simple image with edges
image = np.zeros((10, 10))
image[:, 5:] = 1  # Vertical edge

# Edge detection kernels
sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]])  # Horizontal edges
sobel_y = np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]])  # Vertical edges

edge_x = conv2d(image, sobel_x)
edge_y = conv2d(image, sobel_y)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(edge_x, cmap='gray')
axes[1].set_title('Sobel X (Horizontal Edges)')
axes[1].axis('off')

axes[2].imshow(edge_y, cmap='gray')
axes[2].set_title('Sobel Y (Vertical Edges)')
axes[2].axis('off')

axes[3].imshow(np.abs(edge_x) + np.abs(edge_y), cmap='gray')
axes[3].set_title('Combined Edges')
axes[3].axis('off')

plt.tight_layout()
plt.show()

## 6. CNN Parameter Calculation

In [ ]:
def count_cnn_params(in_channels, out_channels, kernel_size):
    """
    Count parameters in a conv layer
    
    Parameters = kernel_height × kernel_width × in_channels × out_channels + biases
    """
    weights = kernel_size * kernel_size * in_channels * out_channels
    biases = out_channels
    return weights + biases

def count_fc_params(in_features, out_features):
    """Count parameters in a fully connected layer"""
    return in_features * out_features + out_features

# Example: LeNet-5 style architecture
print("=== CNN Parameter Count ===")
print("\nLeNet-5 Style Architecture:")

# Conv1: 1 input channel, 6 output channels, 5×5 kernel
params_conv1 = count_cnn_params(1, 6, 5)
print(f"Conv1: {params_conv1} parameters")

# Conv2: 6 input channels, 16 output channels, 5×5 kernel
params_conv2 = count_cnn_params(6, 16, 5)
print(f"Conv2: {params_conv2} parameters")

# FC1: 16*5*5 inputs, 120 outputs
params_fc1 = count_fc_params(16*5*5, 120)
print(f"FC1: {params_fc1} parameters")

# FC2: 120 inputs, 84 outputs
params_fc2 = count_fc_params(120, 84)
print(f"FC2: {params_fc2} parameters")

# FC3: 84 inputs, 10 outputs
params_fc3 = count_fc_params(84, 10)
print(f"FC3: {params_fc3} parameters")

total = params_conv1 + params_conv2 + params_fc1 + params_fc2 + params_fc3
print(f"\nTotal: {total:,} parameters")

## 7. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: Convolution Output

In [ ]:
# ให้ image = [[1, 2], [3, 4]] และ kernel = [[1, 0], [0, 1]]
# จงคำนวณ convolution output

image = np.array([[1, 2], [3, 4]])
kernel = np.array([[1, 0], [0, 1]])

output = np.sum(image * kernel)
print(f"Image:\n{image}")
print(f"\nKernel:\n{kernel}")
print(f"\nConvolution output: {output}")
print(f"Calculation: 1×1 + 2×0 + 3×0 + 4×1 = {output}")

### แบบฝึกหัดที่ 2: Output Size

In [ ]:
# จงคำนวณ output size ของ:
# - Input: 224×224
# - Kernel: 7×7
# - Stride: 2
# - Padding: 3

input_size = 224
kernel_size = 7
stride = 2
padding = 3

output_size = calculate_output_size(input_size, kernel_size, stride, padding)
print(f"Output size: {output_size}×{output_size}")
print(f"Formula: ({input_size} + 2×{padding} - {kernel_size}) / {stride} + 1 = {output_size}")

### แบบฝึกหัดที่ 3: Max Pooling

In [ ]:
# ให้ feature map = [[1, 3, 2, 4], [5, 6, 7, 8], [9, 2, 1, 3], [4, 5, 6, 7]]
# จงคำนวณ max pooling ด้วย pool_size=2, stride=2

feature_map = np.array([
    [1, 3, 2, 4],
    [5, 6, 7, 8],
    [9, 2, 1, 3],
    [4, 5, 6, 7]
])

pooled = max_pool2d(feature_map, pool_size=2, stride=2)
print(f"Feature map:\n{feature_map}")
print(f"\nMax pooled:\n{pooled}")

### แบบฝึกหัดที่ 4: CNN Parameters

In [ ]:
# จงคำนวณ parameters ของ Conv layer:
# - Input channels: 3 (RGB)
# - Output channels: 64
# - Kernel size: 3×3

in_channels = 3
out_channels = 64
kernel_size = 3

params = count_cnn_params(in_channels, out_channels, kernel_size)
print(f"Conv Layer Parameters:")
print(f"Weights: {kernel_size}×{kernel_size}×{in_channels}×{out_channels} = {kernel_size*kernel_size*in_channels*out_channels}")
print(f"Biases: {out_channels}")
print(f"Total: {params}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **Convolution Operation**: การคำนวณ convolution และ output size
2. **Pooling**: Max pooling และ Average pooling
3. **Edge Detection**: การตรวจจับขอบภาพ
4. **Parameter Calculation**: การคำนวณจำนวน parameters